In [3]:
import pandas as pd
import numpy as np
# import gensim
import pandas as pd
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
messages = pd.read_csv('/content/SMSSpamCollection.txt', sep='\t', names=['label', 'message'])
messages[:2]

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...


In [5]:
corpus = []
for i in range(len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    review = review.lower().split()
    review = [ps.stem(word) for word in review if word not in stopwords.words()]
    review = ' '.join(review)
    corpus.append(review)
corpus[:2]

['jurong point crazi avail bugi great world buffet amor', 'lar joke wif']

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(max_features=2500, ngram_range=(1,2))
X = tv.fit_transform(corpus).toarray()

In [17]:
X.shape

(5572, 2500)

In [79]:
tv.vocabulary_

{'point': np.int64(1581),
 'crazi': np.int64(472),
 'avail': np.int64(116),
 'bugi': np.int64(239),
 'great': np.int64(851),
 'world': np.int64(2433),
 'lar': np.int64(1114),
 'joke': np.int64(1069),
 'wif': np.int64(2389),
 'free': np.int64(746),
 'entri': np.int64(643),
 'wkli': np.int64(2415),
 'comp': np.int64(413),
 'win': np.int64(2392),
 'cup': np.int64(489),
 'final': np.int64(708),
 'tkt': np.int64(2143),
 'st': np.int64(1974),
 'text': np.int64(2098),
 'receiv': np.int64(1697),
 'question': np.int64(1656),
 'std': np.int64(1989),
 'txt': np.int64(2198),
 'rate': np.int64(1673),
 'appli': np.int64(78),
 'free entri': np.int64(753),
 'entri wkli': np.int64(646),
 'wkli comp': np.int64(2416),
 'cup final': np.int64(490),
 'std txt': np.int64(1990),
 'txt rate': np.int64(2206),
 'rate appli': np.int64(1675),
 'dun': np.int64(604),
 'earli': np.int64(612),
 'usf': np.int64(2287),
 'live': np.int64(1171),
 'freemsg': np.int64(766),
 'hey': np.int64(927),
 'darl': np.int64(514),
 'w

In [26]:
y = messages['label'] == 'ham'
y = y.astype('int8')
y.dtype, y.shape

(dtype('int8'), (5572,))

In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [28]:
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [29]:
spam_detect_model.score(X_test, y_test)

0.9757847533632287

In [30]:
from sklearn.metrics import classification_report
print(classification_report(y_test, spam_detect_model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      0.83      0.91       160
           1       0.97      1.00      0.99       955

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.97      1115



In [31]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [43]:
from nltk.corpus import stopwords, wordnet
def get_wordnet_pos(word):
    """Map NLTK POS tag to WordNet POS tag"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

In [54]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [57]:
nltk.pos_tag(['buying'])

[('buying', 'NN')]

In [60]:
nltk.pos_tag(['buying'])[0][1][0]

'N'

In [65]:
cached_stopwords = set(stopwords.words('english'))

def process_text(message):
    review = re.sub('[^a-zA-Z]', ' ', message)
    review = review.lower().split()
    lemmatized_words = []
    for word in review:
        if word not in cached_stopwords:
          pos_tag = get_wordnet_pos(word)
          lemmatized_words.append(lemmatizer.lemmatize(word, pos=pos_tag))
    return ' '.join(lemmatized_words)

In [67]:
messages['message'] = messages['message'].apply(process_text)

In [74]:
lemmatizer.lemmatize('better',pos=nltk.pos_tag(['better'])[0][1][0].lower())

'well'

In [77]:
messages.tail()

,label,message
5567,spam,nd time try contact u u pound prize claim easy...
5568,ham,b go esplanade fr home
5569,ham,pity mood suggestion
5570,ham,guy bitching act like interested buying someth...
5571,ham,rofl true name


In [80]:
corpus_lemmatized = messages['message'].tolist()
len(corpus_lemmatized)

5572

In [81]:
tv = TfidfVectorizer(max_features=2500, ngram_range=(1,2))
X = tv.fit_transform(corpus_lemmatized).toarray()

In [82]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [86]:
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(X_train, y_train)
spam_detect_model.score(X_test, y_test)

0.9775784753363229

In [87]:
from sklearn.metrics import classification_report
print(classification_report(y_test, spam_detect_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.99      0.85      0.92       160
           1       0.98      1.00      0.99       955

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



### BOW, TFIDF, Machine Learning algos

1. Processing and cleaning
2. BOW and TFIDF (Sentence-> vectors)
3. train test split
4. train

we did error split after changing orignal data
so steps 2 and 3 must be interchanged

In [89]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus_lemmatized, y, test_size=0.2, random_state=0)

In [95]:
len(X_train), len(X_test)

(4457, 1115)

## Critical step

In [97]:
X_train = tv.fit_transform(X_train).toarray()
X_test = tv.transform(X_test).toarray()

In [99]:
X_train.shape, X_test.shape

((4457, 2500), (1115, 2500))

In [101]:
spam_detect_model = MultinomialNB().fit(X_train, y_train)
spam_detect_model.score(X_test, y_test)

0.9766816143497757

In [102]:
print(classification_report(y_test, spam_detect_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.99      0.84      0.91       160
           1       0.97      1.00      0.99       955

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

